In [1]:
#Cell 1 Load + Inspect
import os, pickle, numpy as np, torch

SCRIPT_DIR = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code"

def load_pkl(fname):
    path = os.path.join(SCRIPT_DIR, fname)
    with open(path, "rb") as f:
        data = pickle.load(f)
    if hasattr(data, "detach"):
        return data.detach().cpu().numpy()
    return np.array(data)

train_acts = load_pkl("tcnk_train_activation_matrix.pkl")
eval_acts  = load_pkl("tcnk_eval_activation_matrix.pkl")
all_acts   = load_pkl("tcnk_all_activation_matrix.pkl")

print(f"tcnk_train shape: {train_acts.shape}")
print(f"tcnk_eval  shape: {eval_acts.shape}")
print(f"tcnk_all   shape: {all_acts.shape}")

flat = all_acts.flatten()
print(f"\nRaw activation stats:")
print(f"  mean  = {flat.mean():.4f}")
print(f"  std   = {flat.std():.4f}")
print(f"  min   = {flat.min():.4f}")
print(f"  max   = {flat.max():.4f}")
print(f"  > 0   = {(flat > 0).mean()*100:.1f}%")

tcnk_train shape: (63643, 128)
tcnk_eval  shape: (7071, 128)
tcnk_all   shape: (70714, 128)

Raw activation stats:
  mean  = 0.4005
  std   = 0.7618
  min   = 0.0000
  max   = 69.1642
  > 0   = 46.6%


In [2]:
#Cell 2 Normalisation
import torch as th

train_t  = th.tensor(train_acts, dtype=th.float32)
mean_act = train_t.mean(dim=0, keepdim=True)   # shape (1, 128)

train_centered = train_t - mean_act
eval_t  = th.tensor(eval_acts,  dtype=th.float32) - mean_act
all_t   = th.tensor(all_acts,   dtype=th.float32) - mean_act

row_norms = train_centered.norm(dim=1)
scale = th.quantile(row_norms, 0.95).item()
print(f"Normalization scale (95th pctile L2 norm): {scale:.4f}")

train_norm = train_centered / scale
eval_norm  = eval_t  / scale
all_norm   = all_t   / scale

flat_norm = train_norm.numpy().flatten()
print(f"Post-norm stats (train):")
print(f"  mean = {flat_norm.mean():.4f}  (target: ~0)")
print(f"  std  = {flat_norm.std():.4f}   (expect ~0.03-0.06)")
print(f"  max  = {flat_norm.max():.4f}   (target: <5)")

norm_params = {"mean_act": mean_act, "scale": scale}
th.save(norm_params, os.path.join(SCRIPT_DIR, "tcnk_activation_norm_params.pt"))
print("Saved: tcnk_activation_norm_params.pt")

train_np = train_norm.numpy().astype(np.float32)
eval_np  = eval_norm.numpy().astype(np.float32)
all_np   = all_norm.numpy().astype(np.float32)

Normalization scale (95th pctile L2 norm): 8.9932
Post-norm stats (train):
  mean = 0.0000  (target: ~0)
  std  = 0.0631   (expect ~0.03-0.06)
  max  = 7.6665   (target: <5)
Saved: tcnk_activation_norm_params.pt


In [11]:
#Cell 2 Abalation

import sys, os, numpy as np, torch as th, itertools
from torch.utils.data import DataLoader

SCRIPT_DIR = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code"

from dictionary_learning import AutoEncoder
from dictionary_learning.trainers import StandardTrainer
from dictionary_learning.training import trainSAE

device = "cuda" if th.cuda.is_available() else "cpu"
print(f"Import OK  |  device: {device}")

INPUT_DIM = train_np.shape[1]
DICT_SIZE  = INPUT_DIM * 16
print(f"SAE: {INPUT_DIM} -> {DICT_SIZE}  (16x expansion)")
print(f"Train: {len(train_np)}  |  steps=10000 with cycling dataloader")

class _RawDS(th.utils.data.Dataset):
    def __init__(self, t): self.t = t
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i]

def run_ablation(l1_penalty, label):
    train_t_l    = th.tensor(train_np, dtype=th.float32)
    base_loader  = DataLoader(_RawDS(train_t_l), batch_size=32, shuffle=True, drop_last=True)
    # itertools.cycle 让 loader 无限循环，islice 截取恰好 10000 步
    inf_loader   = itertools.islice(itertools.cycle(base_loader), 10000)
    total_steps  = 10000

    save_dir = os.path.join(SCRIPT_DIR, f"sae_tcnk_l1pen{l1_penalty}")
    os.makedirs(save_dir, exist_ok=True)

    trainer_cfg = {
        "trainer":               StandardTrainer,
        "dict_class":            AutoEncoder,
        "activation_dim":        INPUT_DIM,
        "dict_size":             DICT_SIZE,
        "lr":                    1e-3,
        "l1_penalty":            l1_penalty,
        "steps":                 total_steps,
        "warmup_steps":          1000,
        "sparsity_warmup_steps": 2000,
        "resample_steps":        3000,
        "layer":                 1,
        "lm_name":               "tcnk",
        "device":                device,
    }

    trainSAE(
        data=inf_loader,
        trainer_configs=[trainer_cfg],
        steps=total_steps,
        save_dir=save_dir,
        verbose=True,
        log_steps=1000,
    )

    ae = AutoEncoder.from_pretrained(os.path.join(save_dir, "trainer_0/ae.pt"))
    ae.to(device).eval()

    all_t_l = th.tensor(all_np, dtype=th.float32).to(device)
    with th.no_grad():
        recons, features = ae(all_t_l, output_features=True)

    feats_np  = features.cpu().numpy()
    recons_np = recons.cpu().numpy()
    dead      = (feats_np == 0).all(axis=0).sum()
    l0        = (feats_np > 0).sum(axis=1).mean()
    frac_var  = 1.0 - ((all_np - recons_np) ** 2).mean() / (all_np.var() + 1e-10)

    print(f"\n[{label}] l1_penalty={l1_penalty}")
    print(f"  Dead: {dead}/{DICT_SIZE}  L0: {l0:.1f}  frac_var: {frac_var:.4f}")
    return {"l1_penalty": l1_penalty, "dead": int(dead), "l0": float(l0), "frac_var": float(frac_var)}

print("\n=== Ablation 1/3: l1=5e-3 ===")
r1 = run_ablation(5e-3, "1/3")

print("\n=== Ablation 2/3: l1=1e-2 ===")
r2 = run_ablation(1e-2, "2/3")

print("\n=== Ablation 3/3: l1=2e-2 ===")
r3 = run_ablation(2e-2, "3/3")

print("\n=== Summary ===")
for r in [r1, r2, r3]:
    print(f"  l1={r['l1_penalty']:.0e}  dead={r['dead']:4d}  L0={r['l0']:5.1f}  frac_var={r['frac_var']:.4f}")
print("\nTarget: dead~0, L0=20-80, frac_var>0.5")


Import OK  |  device: cpu
SAE: 128 -> 2048  (16x expansion)
Train: 63643  |  steps=10000 with cycling dataloader

=== Ablation 1/3: l1=5e-3 ===


  0%|          | 34/10000 [00:00<00:29, 336.68it/s]

Step 0: L0 = 1026.65625, frac_variance_explained = 0.1515321135520935


 11%|█         | 1085/10000 [00:02<00:18, 489.55it/s]

Step 1000: L0 = 414.21875, frac_variance_explained = 0.9358710646629333


...

Step 8000: L0 = 38.4375, frac_variance_explained = 0.9627829194068909


 91%|█████████ | 9068/10000 [00:19<00:01, 536.59it/s]

Step 9000: L0 = 38.59375, frac_variance_explained = 0.9675706624984741
resampling 997 neurons


100%|██████████| 10000/10000 [00:21<00:00, 470.13it/s]



[3/3] l1_penalty=0.02
  Dead: 938/2048  L0: 44.0  frac_var: 0.9621

=== Summary ===
  l1=5e-03  dead=1789  L0= 54.5  frac_var=0.6351
  l1=1e-02  dead=1753  L0= 44.1  frac_var=0.8046
  l1=2e-02  dead= 938  L0= 44.0  frac_var=0.9621

Target: dead~0, L0=20-80, frac_var>0.5


In [13]:
# Cell 3: 训练最终 T/NK SAE (l1=2e-2, fixed seed)

import sys, os, random, numpy as np, torch as th, itertools
from torch.utils.data import DataLoader

SCRIPT_DIR = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code"
sys.path.insert(0, SCRIPT_DIR)

from dictionary_learning import AutoEncoder
from dictionary_learning.trainers import StandardTrainer
from dictionary_learning.training import trainSAE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)

device    = "cuda" if th.cuda.is_available() else "cpu"
BEST_L1   = 2e-2
INPUT_DIM = train_np.shape[1]
DICT_SIZE = INPUT_DIM * 16
SAVE_DIR  = os.path.join(SCRIPT_DIR, "sae_tcnk_final")
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Final SAE training  |  seed={SEED}  |  l1={BEST_L1}  |  device={device}")
print(f"  {INPUT_DIM} -> {DICT_SIZE}  (16x expansion)")

class _RawDS(th.utils.data.Dataset):
    def __init__(self, t): self.t = t
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i]

train_t     = th.tensor(train_np, dtype=th.float32)
base_loader = DataLoader(_RawDS(train_t), batch_size=32, shuffle=True, drop_last=True)
inf_loader  = itertools.islice(itertools.cycle(base_loader), 10000)
total_steps = 10000

trainer_cfg = {
    "trainer":               StandardTrainer,
    "dict_class":            AutoEncoder,
    "activation_dim":        INPUT_DIM,
    "dict_size":             DICT_SIZE,
    "lr":                    1e-3,
    "l1_penalty":            BEST_L1,
    "steps":                 total_steps,
    "warmup_steps":          1000,
    "sparsity_warmup_steps": 2000,
    "resample_steps":        3000,
    "layer":                 1,
    "lm_name":               "tcnk_final",
    "device":                device,
}

trainSAE(
    data=inf_loader,
    trainer_configs=[trainer_cfg],
    steps=total_steps,
    save_dir=SAVE_DIR,
    verbose=True,
    log_steps=1000,
)

ae_final = AutoEncoder.from_pretrained(os.path.join(SAVE_DIR, "trainer_0/ae.pt"))
ae_final.to(device).eval()

all_t = th.tensor(all_np, dtype=th.float32).to(device)
with th.no_grad():
    recons_f, features_f = ae_final(all_t, output_features=True)

feats_np  = features_f.cpu().numpy()
recons_np = recons_f.cpu().numpy()
dead      = (feats_np == 0).all(axis=0).sum()
l0        = (feats_np > 0).sum(axis=1).mean()
frac_var  = 1.0 - ((all_np - recons_np) ** 2).mean() / (all_np.var() + 1e-10)

print(f"\nFinal model sanity check:")
print(f"  Dead features:           {dead}/{DICT_SIZE}")
print(f"  Active features:         {DICT_SIZE - dead}/{DICT_SIZE}")
print(f"  L0 (mean active/cell):   {l0:.1f}")
print(f"  frac_variance_explained: {frac_var:.4f}")

Final SAE training  |  seed=42  |  l1=0.02  |  device=cpu
  128 -> 2048  (16x expansion)


  0%|          | 45/10000 [00:00<00:22, 444.94it/s]

Step 0: L0 = 1020.75, frac_variance_explained = 0.15330135822296143


 11%|█         | 1090/10000 [00:02<00:16, 544.21it/s]

Step 1000: L0 = 216.0, frac_variance_explained = 0.9536641240119934


...

Step 8000: L0 = 36.25, frac_variance_explained = 0.9579774141311646


 91%|█████████ | 9111/10000 [00:18<00:01, 552.10it/s]

Step 9000: L0 = 34.84375, frac_variance_explained = 0.9487332701683044
resampling 1065 neurons


100%|██████████| 10000/10000 [00:20<00:00, 486.04it/s]



Final model sanity check:
  Dead features:           1438/2048
  Active features:         610/2048
  L0 (mean active/cell):   39.4
  frac_variance_explained: 0.8709


In [14]:
# Cell 4: 提取 T/NK 全量特征矩阵并保存

import sys, os, pickle, numpy as np, torch as th

SCRIPT_DIR = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code"
sys.path.insert(0, SCRIPT_DIR)

from dictionary_learning import AutoEncoder

device   = "cuda" if th.cuda.is_available() else "cpu"
SAVE_DIR = os.path.join(SCRIPT_DIR, "sae_tcnk_final")

ae = AutoEncoder.from_pretrained(os.path.join(SAVE_DIR, "trainer_0/ae.pt"))
ae.to(device).eval()
print(f"Model loaded  |  device: {device}")

all_t = th.tensor(all_np, dtype=th.float32).to(device)

BATCH = 4096
recons_list, feats_list = [], []

with th.no_grad():
    for start in range(0, len(all_t), BATCH):
        batch = all_t[start:start + BATCH]
        r, f  = ae(batch, output_features=True)
        recons_list.append(r.cpu())
        feats_list.append(f.cpu())
        if (start // BATCH) % 5 == 0:
            print(f"  Processed {min(start + BATCH, len(all_t))}/{len(all_t)} cells", flush=True)

recons_all   = th.cat(recons_list, dim=0).numpy()
features_all = th.cat(feats_list,  dim=0).numpy()

print(f"\nfeatures_all shape: {features_all.shape}")   # (70714, 2048)
print(f"recons_all   shape: {recons_all.shape}")

out_path = os.path.join(SCRIPT_DIR, "sae_tcnk_results_all.pkl")
with open(out_path, "wb") as f:
    pickle.dump((recons_all, features_all), f)

print(f"\nSaved: {out_path}")
print(f"Active features: {(features_all > 0).any(axis=0).sum()}/2048")
print(f"Mean L0: {(features_all > 0).sum(axis=1).mean():.1f}")

Model loaded  |  device: cpu
  Processed 4096/70714 cells
  Processed 24576/70714 cells
  Processed 45056/70714 cells
  Processed 65536/70714 cells

features_all shape: (70714, 2048)
recons_all   shape: (70714, 128)

Saved: /Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code/sae_tcnk_results_all.pkl
Active features: 610/2048
Mean L0: 39.4
